# Experimental Qwen2.5-0.5B LoRA competition submission at 1,024 tokens
Uses the previously trained private LoRA adapter. No training, no Internet. Context 1,024 was selected after the exploratory same-validation-set ablation (local log loss 1.320317 vs 1.532768 at 768 and 1.964393 at 384). This is an authorized experimental Kaggle submission, not a claim that it beats the existing length-only 1.06530 entry.

**Version 2 repair:** bundle `length_baseline.py`, required indirectly by `finetune_lora.py`. Version 1 failed before model inference with `ModuleNotFoundError` and was never submitted.


In [ ]:
from pathlib import Path
import importlib, importlib.metadata, subprocess, sys
from packaging.version import Version
try:
    v=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    v=None
if v is not None and Version(v)<=Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
    importlib.invalidate_caches()
import torch
if not torch.cuda.is_available(): raise RuntimeError('GPU required')
root=Path('/kaggle/input')
mounted=sorted(p.name for p in root.iterdir()) if root.is_dir() else []
csv_roots=sorted({p.parent for p in root.rglob('test.csv')})
competition=[p for p in csv_roots if 'llm-classification-finetuning' in str(p).lower()]
if not competition and len(csv_roots)==1: competition=csv_roots
if len(competition)!=1: raise FileNotFoundError('Competition test.csv mount not uniquely found: '+repr(mounted))
TEST=competition[0]/'test.csv'
base=[p.parent for p in root.rglob('config.json') if 'qwen2.5' in str(p).lower() and '0.5b' in str(p).lower()]
if len(base)!=1: raise FileNotFoundError('Qwen base mount not uniquely found: '+repr(mounted))
BASE=base[0]
adapters=[p.parent for p in root.rglob('adapter_model.safetensors') if (p.parent/'adapter_config.json').is_file()]
previous=[p for p in adapters if 'llm-preference-qwen05b-lora-pilot' in str(p)]
if not previous and len(adapters)==1: previous=adapters
if len(previous)!=1: raise FileNotFoundError('Saved Qwen LoRA adapter not uniquely found: '+repr(mounted))
ADAPTER=previous[0]
print('test',TEST,'base',BASE,'adapter',ADAPTER)


In [ ]:
# Generated source bundle for offline Kaggle submission.
from pathlib import Path
import sys
source_dir=Path('/kaggle/working/src')
source_dir.mkdir(parents=True,exist_ok=True)
(source_dir/'__init__.py').write_text('',encoding='utf-8')
(source_dir/'baseline.py').write_text("\"\"\"Leakage-controlled, swap-augmented TF-IDF baseline for Kaggle LLM preference prediction.\n\nThis is a classical ML baseline, not an LLM fine-tuning run.\n\"\"\"\nimport argparse\nimport ast\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack, vstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nTARGETS = [\"winner_model_a\", \"winner_model_b\", \"winner_tie\"]\nTEXT_COLUMNS = [\"prompt\", \"response_a\", \"response_b\"]\n\n\ndef flatten_messages(value, max_chars=2400):\n    \"\"\"Normalize Kaggle's serialized lists of turns; cap length for a CPU starter.\"\"\"\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return \"\"\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith(\"[\"):\n            try:\n                value = json.loads(text)\n            except (ValueError, TypeError):\n                try:\n                    value = ast.literal_eval(text)\n                except (ValueError, SyntaxError):\n                    value = text\n        else:\n            value = text\n    if isinstance(value, (list, tuple)):\n        text = \" \".join(\"\" if item is None else str(item) for item in value)\n    else:\n        text = str(value)\n    return text[:max_chars]\n\n\ndef normalized_frame(df):\n    missing = [c for c in TEXT_COLUMNS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing text columns: {missing}\")\n    return pd.DataFrame(\n        {col: [flatten_messages(v) for v in df[col]] for col in TEXT_COLUMNS},\n        index=df.index,\n    )\n\n\ndef get_labels(df):\n    missing = [c for c in TARGETS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing label columns: {missing}\")\n    y = df[TARGETS].to_numpy(dtype=int)\n    if not np.all(y.sum(axis=1) == 1) or not np.all((y == 0) | (y == 1)):\n        raise ValueError(\"Expected exactly one binary winner label per training row\")\n    return y.argmax(axis=1)\n\n\ndef flip_pairs(df):\n    flipped = df.copy()\n    flipped[\"response_a\"], flipped[\"response_b\"] = (\n        df[\"response_b\"].copy(), df[\"response_a\"].copy()\n    )\n    return flipped\n\n\ndef make_vectorizer(df):\n    # Only fit on training-partition texts; do not fit on held-out validation/test.\n    min_df = 2 if len(df) >= 30 else 1\n    vectorizer = TfidfVectorizer(\n        ngram_range=(1, 2), max_features=35000, min_df=min_df,\n        strip_accents=\"unicode\", sublinear_tf=True, dtype=np.float32,\n    )\n    vectorizer.fit(\n        df[\"prompt\"].tolist() + df[\"response_a\"].tolist() +\n        df[\"response_b\"].tolist()\n    )\n    return vectorizer\n\n\ndef pair_features(df, vectorizer):\n    q = vectorizer.transform(df[\"prompt\"])\n    a = vectorizer.transform(df[\"response_a\"])\n    b = vectorizer.transform(df[\"response_b\"])\n    len_a = df[\"response_a\"].str.len().to_numpy(dtype=np.float32)\n    len_b = df[\"response_b\"].str.len().to_numpy(dtype=np.float32)\n    len_q = df[\"prompt\"].str.len().to_numpy(dtype=np.float32)\n    numeric = np.column_stack([\n        np.log1p(len_a) - np.log1p(len_b),\n        (np.log1p(len_a) + np.log1p(len_b)) / 2,\n        np.log1p(len_q),\n    ]) / 10.0\n    return hstack([q, a - b, (a + b) * 0.5, csr_matrix(numeric)],\n                  format=\"csr\", dtype=np.float32)\n\n\ndef fit_baseline(df, y):\n    vectorizer = make_vectorizer(df)\n    x_original = pair_features(df, vectorizer)\n    x_flipped = pair_features(flip_pairs(df), vectorizer)\n    swapped_labels = np.where(y == 0, 1, np.where(y == 1, 0, 2))\n    model = LogisticRegression(C=2.0, max_iter=300, random_state=42)\n    model.fit(vstack([x_original, x_flipped], format=\"csr\"),\n              np.concatenate([y, swapped_labels]))\n    return {\"vectorizer\": vectorizer, \"model\": model, \"targets\": TARGETS}\n\n\ndef predict_prob(bundle, df):\n    features = pair_features(df, bundle[\"vectorizer\"])\n    raw = bundle[\"model\"].predict_proba(features)\n    out = np.zeros((len(df), len(TARGETS)), dtype=np.float64)\n    for col_idx, class_idx in enumerate(bundle[\"model\"].classes_):\n        out[:, int(class_idx)] = raw[:, col_idx]\n    return out / out.sum(axis=1, keepdims=True)\n\n\ndef train(train_csv, out_dir, validation_fraction=0.15):\n    raw = pd.read_csv(train_csv)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if len(np.unique(y)) != 3 or np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class must have at least two examples for validation\")\n    x_tr, x_val, y_tr, y_val = train_test_split(\n        df, y, test_size=validation_fraction, random_state=42, stratify=y\n    )\n    validation_bundle = fit_baseline(x_tr, y_tr)\n    val_probs = predict_prob(validation_bundle, x_val)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_probs, labels=[0, 1, 2])),\n        \"train_rows\": int(len(x_tr)),\n        \"validation_rows\": int(len(x_val)),\n        \"full_rows\": int(len(df)),\n        \"seed\": 42,\n        \"note\": \"Random stratified split; not a competition leaderboard result.\",\n    }\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \"validation_metrics.json\").write_text(\n        json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    # Refit using all labeled data only after holding out validation above.\n    joblib.dump(fit_baseline(df, y), output / \"baseline.joblib\")\n    return metrics\n\n\ndef predict(test_csv, model_path, out_csv):\n    test = pd.read_csv(test_csv)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Test CSV must contain id\")\n    bundle = joblib.load(model_path)  # Load only artifacts you created/trust.\n    probabilities = predict_prob(bundle, normalized_frame(test))\n    result = pd.DataFrame(probabilities, columns=TARGETS)\n    result.insert(0, \"id\", test[\"id\"])\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out_csv, index=False)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\"command\", required=True)\n    fit = sub.add_parser(\"train\")\n    fit.add_argument(\"--train\", default=\"data/train.csv\")\n    fit.add_argument(\"--out\", default=\"artifacts\")\n    infer = sub.add_parser(\"predict\")\n    infer.add_argument(\"--test\", default=\"data/test.csv\")\n    infer.add_argument(\"--model\", default=\"artifacts/baseline.joblib\")\n    infer.add_argument(\"--out\", default=\"submission.csv\")\n    args = parser.parse_args()\n    if args.command == \"train\":\n        print(json.dumps(train(args.train, args.out), indent=2))\n    else:\n        print(f\"Wrote {len(predict(args.test, args.model, args.out))} rows: {args.out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",encoding='utf-8')
(source_dir/'length_baseline.py').write_text("\"\"\"Nested-tuned, length-only A/B/tie baseline, exploratory validation.\n\nThis tests whether simple observable character lengths provide predictive signal.\nIt neither measures warmth nor establishes a causal preference for verbosity.\nOnly aggregate JSON may be published; official Kaggle data stay local.\n\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with open(path, 'rb') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b''):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef feature_matrix(df):\n    \"\"\"No semantic word features: lengths of complete joined conversation fields.\"\"\"\n    def lengths(col):\n        return np.asarray(\n            [len(flatten_messages(value, max_chars=2_000_000)) for value in df[col]],\n            dtype=np.float64,\n        )\n    a = np.log1p(lengths(\"response_a\"))\n    b = np.log1p(lengths(\"response_b\"))\n    q = np.log1p(lengths(\"prompt\"))\n    delta = a - b\n    return np.column_stack([\n        delta,\n        np.abs(delta),\n        (a + b) / 2,\n        q,\n        delta * (q / 10),\n        (a - b) / np.maximum((a + b), 1.0),\n    ])\n\n\ndef fit_length_model(frame, labels, c=1.0):\n    original = feature_matrix(frame)\n    swapped = feature_matrix(flip_pairs(frame))\n    swapped_labels = np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n    model = LogisticRegression(C=c, max_iter=700, random_state=42)\n    model.fit(\n        np.vstack([original, swapped]),\n        np.concatenate([labels, swapped_labels])\n    )\n    return model\n\n\ndef predict_length_model(model, frame):\n    raw = model.predict_proba(feature_matrix(frame))\n    out = np.zeros((len(frame), 3), dtype=np.float64)\n    for idx, label in enumerate(model.classes_):\n        out[:, int(label)] = raw[:, idx]\n    return out\n\n\ndef logloss_delta_interval(labels, prediction, reference, draws=1000, seed=42):\n    \"\"\"Bootstrap per-row excess log loss; negative means model lower loss.\"\"\"\n    y = np.asarray(labels, dtype=np.int64)\n    selected_model = np.clip(prediction[np.arange(len(y)), y], 1e-15, 1)\n    selected_reference = np.clip(reference[np.arange(len(y)), y], 1e-15, 1)\n    delta = -np.log(selected_model) + np.log(selected_reference)\n    rng = np.random.default_rng(seed)\n    draws_arr = np.array([\n        delta[rng.integers(0, len(delta), len(delta))].mean()\n        for _ in range(draws)\n    ])\n    return [float(x) for x in np.quantile(draws_arr, [0.025, 0.975])]\n\n\ndef run_pilot(train_csv, out_dir, sample_size=12000, seed=42):\n    source = Path(train_csv)\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = pd.read_csv(source)\n    labels = get_labels(raw)\n    if len(raw) < 100 or np.min(np.bincount(labels, minlength=3)) < 10:\n        raise ValueError(\"Need >=100 labeled examples and >=10 per class\")\n    if sample_size < 0 or (sample_size > 0 and sample_size < 100):\n        raise ValueError(\"Pilot size should be 0 for all rows or >=100\")\n    if sample_size and sample_size < len(raw):\n        chosen, _ = train_test_split(\n            np.arange(len(raw)), train_size=sample_size,\n            stratify=labels, random_state=seed,\n        )\n        pilot = raw.iloc[chosen].reset_index(drop=True)\n    else:\n        pilot = raw.reset_index(drop=True)\n    pilot_y = get_labels(pilot)\n    if np.min(np.bincount(pilot_y, minlength=3)) < 10:\n        raise ValueError(\n            'Sampled pilot must contain at least 10 rows per class for nested splitting'\n        )\n    # Exactly the outer split used in the 2026-09-23 TF-IDF benchmark.\n    outer_train, outer_val, y_train, y_val = train_test_split(\n        pilot, pilot_y, test_size=0.15, random_state=42, stratify=pilot_y\n    )\n    # Hyperparameter selection must be based only on an INNER calibration fold.\n    inner_train, inner_val, inner_y, inner_val_y = train_test_split(\n        outer_train, y_train, test_size=0.20,\n        random_state=1337, stratify=y_train\n    )\n    candidate_c = [0.01, 0.1, 1.0, 10.0]\n    inner_results = {}\n    for c in candidate_c:\n        model = fit_length_model(inner_train, inner_y, c=c)\n        inner_results[str(c)] = float(\n            log_loss(inner_val_y, predict_length_model(model, inner_val),\n                     labels=[0, 1, 2])\n        )\n    best_c = min(candidate_c, key=lambda c: inner_results[str(c)])\n    model = fit_length_model(outer_train, y_train, c=best_c)\n    prediction = predict_length_model(model, outer_val)\n    uniform = np.full_like(prediction, 1 / 3)\n    prior = np.bincount(y_train, minlength=3).astype(np.float64)\n    prior /= prior.sum()\n    prior_probs = np.tile(prior, (len(y_val), 1))\n    metrics = {\n        \"source\":\"Official Kaggle LLM Classification Finetuning training CSV\",\n        \"source_file_sha256\":file_sha256(source),\n        \"official_training_rows\":int(len(raw)),\n        \"pilot_rows\":int(len(pilot)),\n        \"outer_validation_rows\":int(len(y_val)),\n        \"outer_validation_log_loss_length_only\":float(log_loss(y_val, prediction,labels=[0,1,2])),\n        \"outer_validation_log_loss_training_prior\":float(log_loss(y_val, prior_probs,labels=[0,1,2])),\n        \"outer_validation_log_loss_uniform\":float(log_loss(y_val, uniform,labels=[0,1,2])),\n        \"length_minus_prior_log_loss_bootstrap_95pct\":logloss_delta_interval(\n            y_val, prediction, prior_probs, seed=seed,\n        ),\n        \"inner_validation_log_loss_by_c\":inner_results,\n        \"selected_c\":best_c,\n        \"outer_seed\":42,\n        \"inner_seed\":1337,\n        \"pilot_seed\":seed,\n        \"note\":\"EXPLORATORY comparison. Outer fold overlaps initial TF-IDF benchmark; do not reuse it indefinitely for model selection. Not a Kaggle submission.\",\n        \"constraints\":[\"Row-random splitting; repeated prompts may cross splits.\",\"Character lengths are not measures of style or warmth.\",\"Selected C tuned on inner fold only.\",\"No text features: performance reflects length correlates, not causal effects.\"],\n        \"environment\":{\n            \"scikit_learn\":sklearn.__version__,\n            \"pandas\":pd.__version__,\n            \"github_sha\":os.getenv(\"GITHUB_SHA\",\"local\"),\n            \"run_utc\":datetime.now(timezone.utc).isoformat(),\n        }\n    }\n    (output/\"summary.json\").write_text(\n        json.dumps(metrics,indent=2)+\"\\n\",encoding=\"utf-8\"\n    )\n    print(\"Official training rows:\",metrics[\"official_training_rows\"])\n    print(\"Pilot training rows:\",metrics[\"pilot_rows\"])\n    print(\"Chosen inner-fold regularization C:\",best_c)\n    print(\"Outer length-only log loss:\",metrics[\"outer_validation_log_loss_length_only\"])\n    print(\"Outer train-prior log loss:\",metrics[\"outer_validation_log_loss_training_prior\"])\n    print(\"95% bootstrap (length minus prior):\",metrics[\"length_minus_prior_log_loss_bootstrap_95pct\"])\n    print(\"Aggregate summary saved.\")\n    return metrics\n\n\nif __name__ == \"__main__\":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\",default=\"data/train.csv\")\n    p.add_argument(\"--out\",default=\"artifacts/official_length_pilot\")\n    p.add_argument(\"--sample-size\",type=int,default=12000)\n    a=p.parse_args()\n    run_pilot(a.train,a.out,a.sample_size)\n",encoding='utf-8')
(source_dir/'finetune_lora.py').write_text("\"\"\"GPU pilot: fine-tune an offline Qwen2.5-0.5B base sequence classifier with LoRA.\n\nOnly run after attaching legitimately accessible model weights and official Kaggle\ncompetition data. This is a pilot; no actual GPU experiment is claimed here.\n\nWith offline Kaggle submissions, attach the *complete* base model directory\n(config, tokenizer and weights) as an Input; never fetch from Hugging Face at runtime.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels, normalized_frame\nfrom src.length_baseline import fit_length_model, predict_length_model\n\n\ndef render_pair(row):\n    \"\"\"Fixed prompt template and truncation; do not include train-only model names.\"\"\"\n    question = flatten_messages(row[\"prompt\"], max_chars=1200)\n    a = flatten_messages(row[\"response_a\"], max_chars=2400)\n    b = flatten_messages(row[\"response_b\"], max_chars=2400)\n    return (\n        \"A human gave two chatbots the same user request.\\n\"\n        f\"User request: {question}\\n\"\n        f\"Response A: {a}\\n\"\n        f\"Response B: {b}\\n\"\n        \"Predict whether the human prefers response A, response B, or a tie.\"\n    )\n\n\ndef swap_labels(labels):\n    labels = np.asarray(labels, dtype=np.int64)\n    if not np.isin(labels, [0, 1, 2]).all():\n        raise ValueError(\"Expected A=0, B=1, tie=2\")\n    return np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n\n\ndef train_and_predict(args):\n    # Lazy imports ensure that unit tests do not need GPU-only dependencies.\n    import torch\n    from peft import LoraConfig, TaskType, get_peft_model\n    from torch.utils.data import Dataset\n    from transformers import (\n        AutoModelForSequenceClassification, AutoTokenizer,\n        DataCollatorWithPadding, Trainer, TrainingArguments,\n    )\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"This LoRA pilot requires a CUDA GPU; use the CPU baseline otherwise.\")\n    model_dir = Path(args.base_model).expanduser()\n    if not (model_dir / \"config.json\").exists():\n        raise FileNotFoundError(\n            \"Supply the complete offline base model directory via --base-model; \"\n            f\"no config.json found in {model_dir}\"\n        )\n    if args.pilot_rows != 0 and args.pilot_rows < 30:\n        raise ValueError(\"pilot_rows must be 0 (all rows) or at least 30\")\n\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    raw = pd.read_csv(args.train)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class needs at least two rows\")\n    x_train, x_val, y_train, y_val = train_test_split(\n        df, y, test_size=0.15, stratify=y, random_state=args.seed\n    )\n    # Select an optional, stratified validation subset *before* any GPU training.\n    # The complete official split remains unmodified in the source data.\n    max_validation_rows = getattr(args, \"max_validation_rows\", 0)\n    if max_validation_rows and len(x_val) > max_validation_rows:\n        x_val, _, y_val, _ = train_test_split(\n            x_val, y_val, train_size=max_validation_rows,\n            stratify=y_val, random_state=args.seed\n        )\n    # Cap *training only*. Keep validation untouched by augmentation.\n    if args.pilot_rows and len(x_train) > args.pilot_rows:\n        x_train, _, y_train, _ = train_test_split(\n            x_train, y_train, train_size=args.pilot_rows,\n            stratify=y_train, random_state=args.seed\n        )\n    x_original, y_original = x_train.copy(), y_train.copy()\n    # A fair, matched, same-training-size reference for the GPU pilot.\n    length_reference = fit_length_model(x_original, y_original, c=10.0)\n    length_reference_prob = predict_length_model(length_reference, x_val)\n    matched_length_loss = float(log_loss(y_val, length_reference_prob, labels=[0, 1, 2]))\n    if args.swap_train:\n        x_train = pd.concat(\n            [x_original, flip_pairs(x_original)], ignore_index=True\n        )\n        y_train = np.concatenate([y_original, swap_labels(y_original)])\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        model_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer has neither pad nor EOS token\")\n        tokenizer.pad_token = tokenizer.eos_token\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_dir, num_labels=3, torch_dtype=dtype,\n        local_files_only=True, trust_remote_code=False,\n    )\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    model = get_peft_model(\n        model,\n        LoraConfig(\n            task_type=TaskType.SEQ_CLS,\n            r=8, lora_alpha=16, lora_dropout=0.05,\n            target_modules=[\"q_proj\", \"v_proj\"],\n            modules_to_save=[\"score\"],\n        ),\n    )\n\n    class PairDataset(Dataset):\n        def __init__(self, frame, labels=None):\n            self.texts = [render_pair(row) for row in frame.to_dict(\"records\")]\n            self.labels = labels\n\n        def __len__(self):\n            return len(self.texts)\n\n        def __getitem__(self, i):\n            encoded = tokenizer(\n                self.texts[i], truncation=True, max_length=args.max_length\n            )\n            if self.labels is not None:\n                encoded[\"labels\"] = int(self.labels[i])\n            return encoded\n\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    config = TrainingArguments(\n        output_dir=str(output / \"trainer\"),\n        num_train_epochs=args.epochs,\n        per_device_train_batch_size=args.batch_size,\n        per_device_eval_batch_size=args.eval_batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        weight_decay=0.01,\n        lr_scheduler_type=\"cosine\",\n        warmup_ratio=0.05,\n        fp16=dtype == torch.float16,\n        bf16=dtype == torch.bfloat16,\n        eval_strategy=\"no\",\n        save_strategy=\"no\",\n        logging_strategy=\"steps\",\n        logging_steps=25,\n        report_to=\"none\",\n        remove_unused_columns=False,\n        dataloader_num_workers=0,\n        seed=args.seed,\n    )\n    trainer = Trainer(\n        model=model,\n        args=config,\n        train_dataset=PairDataset(x_train, y_train),\n        data_collator=DataCollatorWithPadding(\n            tokenizer=tokenizer, pad_to_multiple_of=8\n        ),\n        processing_class=tokenizer,\n    )\n    trainer.train()\n    val_logits = trainer.predict(PairDataset(x_val)).predictions\n    if isinstance(val_logits, tuple):\n        val_logits = val_logits[0]\n    val_prob = softmax(np.asarray(val_logits, dtype=np.float64), axis=-1)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_prob, labels=[0, 1, 2])),\n        \"validation_rows\": len(x_val),\n        \"train_rows_after_augmentation\": len(x_train),\n        \"pilot_rows\": args.pilot_rows,\n        \"seed\": args.seed,\n        \"max_length_tokens\": args.max_length,\n        \"base_model_dir\": model_dir.name,\n        \"training_type\": \"Qwen2.5-0.5B base sequence classification head + LoRA\",\n        \"matched_length_reference_log_loss\": matched_length_loss,\n        \"reference_note\": \"Length-only C=10 refit on exactly the GPU pilot original training subset; identical held-out validation rows.\",\n        \"caution\": \"Preliminary pilot; independent replication and full-data run pending.\",\n    }\n    # Probe original/swap consistency on a bounded held-out subset.\n    subset = x_val.iloc[: min(128, len(x_val))]\n    original_logits = trainer.predict(PairDataset(subset)).predictions\n    swapped_logits = trainer.predict(PairDataset(flip_pairs(subset))).predictions\n    if isinstance(original_logits, tuple):\n        original_logits = original_logits[0]\n    if isinstance(swapped_logits, tuple):\n        swapped_logits = swapped_logits[0]\n    original_prob = softmax(np.asarray(original_logits, dtype=np.float64), axis=-1)\n    swapped_prob = softmax(np.asarray(swapped_logits, dtype=np.float64), axis=-1)[:, [1, 0, 2]]\n    metrics[\"swap_probe_mean_abs_difference\"] = float(\n        np.abs(original_prob - swapped_prob).mean()\n    )\n    # Preserve successful validation metrics even if adapter saving or optional\n    # 25K-row Kaggle test inference fails. The JSON is aggregate-only.\n    metrics[\"pipeline_status\"] = \"validation_completed\"\n    try:\n        adapter_dir = output / \"adapter\"\n        trainer.model.save_pretrained(adapter_dir)\n        tokenizer.save_pretrained(adapter_dir)\n        metrics[\"pipeline_status\"] = \"adapter_saved\"\n\n        if args.test:\n            test = pd.read_csv(args.test)\n            if \"id\" not in test.columns:\n                raise ValueError(\"Test data require id column\")\n            test_frame = normalized_frame(test)\n            test_logits = trainer.predict(PairDataset(test_frame)).predictions\n            if isinstance(test_logits, tuple):\n                test_logits = test_logits[0]\n            probs = softmax(np.asarray(test_logits, dtype=np.float64), axis=-1)\n            submission = pd.DataFrame(probs, columns=TARGETS)\n            submission.insert(0, \"id\", test[\"id\"])\n            submission_path = Path(args.submission)\n            submission_path.parent.mkdir(parents=True, exist_ok=True)\n            submission.to_csv(submission_path, index=False)\n            metrics[\"submission_rows\"] = int(len(submission))\n            metrics[\"pipeline_status\"] = \"submission_completed\"\n            print(f\"Submission written to {submission_path}\")\n    except Exception as exc:\n        metrics[\"pipeline_status\"] = \"downstream_failed\"\n        metrics[\"downstream_error_type\"] = type(exc).__name__\n        raise\n    finally:\n        (output / \"gpu_pilot_metrics.json\").write_text(\n            json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n        )\n    print(json.dumps(metrics, indent=2))\n    return metrics\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\", default=\"data/train.csv\")\n    p.add_argument(\"--test\", default=None)\n    p.add_argument(\"--base-model\", required=True,\n                   help=\"Complete *local* Qwen2.5-0.5B weights/tokenizer folder\")\n    p.add_argument(\"--output\", default=\"artifacts/gpu_pilot\")\n    p.add_argument(\"--submission\", default=\"submission.csv\")\n    p.add_argument(\"--pilot-rows\", type=int, default=4000,\n                   help=\"Training cap excluding held-out validation; 0 uses all train rows\")\n    p.add_argument(\"--max-length\", type=int, default=384)\n    p.add_argument(\"--max-validation-rows\", type=int, default=1200,\n                   help=\"Stratified subset of held-out validation for pilot runtime; 0 uses all\")\n    p.add_argument(\"--epochs\", type=float, default=1.0)\n    p.add_argument(\"--batch-size\", type=int, default=2)\n    p.add_argument(\"--eval-batch-size\", type=int, default=4)\n    p.add_argument(\"--grad-accum\", type=int, default=8)\n    p.add_argument(\"--learning-rate\", type=float, default=2e-4)\n    p.add_argument(\"--seed\", type=int, default=42)\n    p.add_argument(\"--no-swap-train\", action=\"store_false\", dest=\"swap_train\")\n    p.set_defaults(swap_train=True)\n    return p.parse_args()\n\n\nif __name__ == \"__main__\":\n    train_and_predict(parse_args())\n",encoding='utf-8')
(source_dir/'qwen_submit_1024.py').write_text("\"\"\"Competition inference using the EXISTING saved Qwen2.5-0.5B LoRA adapter at 1024 tokens.\n\nNo training occurs here. This is an explicitly authorized experimental Kaggle\nsubmission. The notebook stays private, Internet-off, and writes only submission.csv.\n\"\"\"\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\n\nfrom src.baseline import TARGETS, normalized_frame\nfrom src.finetune_lora import render_pair\n\n\ndef predict_batches(model, tokenizer, frame, device=\"cuda:0\", max_length=1024, batch_size=2):\n    import torch\n    if len(frame)==0:\n        raise ValueError(\"Test frame is empty\")\n    model.eval()\n    out=[]\n    with torch.inference_mode():\n        for start in range(0,len(frame),batch_size):\n            chunk=frame.iloc[start:start+batch_size]\n            texts=[render_pair(row) for row in chunk.to_dict(\"records\")]\n            enc=tokenizer(texts,truncation=True,max_length=max_length,padding=True,return_tensors=\"pt\")\n            batch={k:v.to(device) for k,v in enc.items()}\n            logits=model(**batch).logits.detach().float().cpu().numpy()\n            if logits.ndim!=2 or logits.shape[1]!=3:\n                raise ValueError(f\"Expected Nx3 logits, got {logits.shape}\")\n            out.append(softmax(logits.astype(np.float64),axis=1))\n    probs=np.vstack(out)\n    probs=np.clip(probs,1e-12,1.0)\n    return probs/probs.sum(axis=1,keepdims=True)\n\n\ndef run(args):\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    base_dir=Path(args.base_model)\n    adapter_dir=Path(args.adapter)\n    if not (base_dir/\"config.json\").is_file():\n        raise FileNotFoundError(\"Missing Qwen base config\")\n    if not (adapter_dir/\"adapter_config.json\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter config\")\n    if not (adapter_dir/\"adapter_model.safetensors\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter weights\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Kaggle GPU is required for this code submission\")\n\n    test=pd.read_csv(args.test)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Competition test.csv requires id\")\n    frame=normalized_frame(test)\n\n    tok=AutoTokenizer.from_pretrained(base_dir,local_files_only=True,trust_remote_code=False)\n    if tok.pad_token_id is None:\n        if tok.eos_token is None:\n            raise ValueError(\"Tokenizer lacks pad and EOS\")\n        tok.pad_token=tok.eos_token\n    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    base=AutoModelForSequenceClassification.from_pretrained(\n        base_dir,num_labels=3,torch_dtype=dtype,local_files_only=True,trust_remote_code=False\n    )\n    base.config.pad_token_id=tok.pad_token_id\n    base.config.use_cache=False\n    model=PeftModel.from_pretrained(base,adapter_dir,is_trainable=False,local_files_only=True)\n    model.to(\"cuda:0\")\n    probs=predict_batches(model,tok,frame,max_length=args.max_length,batch_size=args.batch_size)\n    sub=pd.DataFrame(probs,columns=TARGETS)\n    sub.insert(0,\"id\",test[\"id\"])\n    out=Path(args.output)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    sub.to_csv(out,index=False)\n    print(f\"Wrote {len(sub)} rows to {out}\")\n    print(\"Context tokens:\",args.max_length,\"batch size:\",args.batch_size)\n    return len(sub)\n\n\nif __name__==\"__main__\":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--test\",required=True)\n    p.add_argument(\"--base-model\",required=True)\n    p.add_argument(\"--adapter\",required=True)\n    p.add_argument(\"--output\",default=\"/kaggle/working/submission.csv\")\n    p.add_argument(\"--max-length\",type=int,default=1024)\n    p.add_argument(\"--batch-size\",type=int,default=2)\n    run(p.parse_args())\n",encoding='utf-8')
sys.path.insert(0,'/kaggle/working')
from src.qwen_submit_1024 import run


In [ ]:
from argparse import Namespace
args=Namespace(test=str(TEST),base_model=str(BASE),adapter=str(ADAPTER),output='/kaggle/working/submission.csv',max_length=1024,batch_size=2)
rows=run(args)
print('Submission rows:',rows)
